# IMS Anomaly Validation

This notebook validates whether anomaly alerts correspond to visible degradation before any application integration.

Validation criteria:

- Alerts should appear late in the experiment lifecycle.
- Alerts should concentrate on documented failed bearings when that information is available.
- Alerts should coincide with elevated degradation features versus the early healthy baseline.
- Alerts are more trustworthy when they are persistent or clustered, not isolated single points.

The current baseline score file evaluates the later chronological portion of `1st_test`, whose documented failures are bearing 3 inner race defect and bearing 4 roller element defect.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_PATH = PROJECT_ROOT / "data" / "processed" / "ims_features.csv"
SCORE_PATH = PROJECT_ROOT / "data" / "processed" / "ims_anomaly_baseline_scores.csv"
VALIDATED_ALERTS_PATH = PROJECT_ROOT / "data" / "processed" / "ims_validated_alerts.csv"
VALIDATION_SUMMARY_PATH = PROJECT_ROOT / "data" / "processed" / "ims_anomaly_validation_summary.csv"
PLOTS_DIR = PROJECT_ROOT / "artifacts" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_EXPERIMENT = "1st_test"
FAILED_BEARINGS = {3, 4}
HEALTHY_FRACTION = 0.2
LATE_LIFE_FRACTION = 0.8
EVIDENCE_QUANTILE = 0.95
PERSISTENCE_WINDOW = 3
METRICS = ["rms", "kurtosis", "crest_factor", "spectral_energy"]

sns.set_theme(style="whitegrid")
FEATURE_PATH, SCORE_PATH

## 1. Load Features and Alerts

In [ ]:
features = pd.read_csv(FEATURE_PATH, parse_dates=["timestamp"])
scores = pd.read_csv(SCORE_PATH, parse_dates=["timestamp"])

features = features.sort_values(["experiment", "timestamp", "sensor_channel"]).reset_index(drop=True)
scores = scores.sort_values(["experiment", "timestamp", "sensor_channel"]).reset_index(drop=True)

test_features = features[features["experiment"].eq(BASELINE_EXPERIMENT)].copy()
timestamps = test_features[["timestamp"]].drop_duplicates().sort_values("timestamp").reset_index(drop=True)
timestamps["full_time_index"] = np.arange(len(timestamps))
timestamps["full_life_fraction"] = timestamps["full_time_index"] / max(len(timestamps) - 1, 1)

test_features = test_features.merge(timestamps, on="timestamp", how="left")
scores = scores.merge(timestamps[["timestamp", "full_life_fraction"]], on="timestamp", how="left")

score_inventory = pd.DataFrame(
    [
        {
            "score_rows": len(scores),
            "score_timestamps": scores["timestamp"].nunique(),
            "feature_rows_for_experiment": len(test_features),
            "feature_timestamps_for_experiment": test_features["timestamp"].nunique(),
            "documented_failed_bearings": ", ".join(str(item) for item in sorted(FAILED_BEARINGS)),
        }
    ]
)

score_inventory

## 2. Healthy Baseline Feature Thresholds

In [ ]:
healthy_features = test_features[test_features["full_life_fraction"].le(HEALTHY_FRACTION)].copy()

threshold_rows = []
for channel, channel_frame in healthy_features.groupby("sensor_channel"):
    for metric in METRICS:
        threshold_rows.append(
            {
                "sensor_channel": int(channel),
                "metric": metric,
                "healthy_median": float(channel_frame[metric].median()),
                "healthy_p95": float(channel_frame[metric].quantile(EVIDENCE_QUANTILE)),
            }
        )

healthy_thresholds = pd.DataFrame(threshold_rows)
healthy_thresholds.head(12)

## 3. Build Alert Evidence Table

In [ ]:
score_feature_columns = ["timestamp", "experiment", "sensor_channel", "bearing", "axis", "rms", "kurtosis", "crest_factor", "spectral_energy"]
scored_features = scores.merge(
    test_features[score_feature_columns],
    on=["timestamp", "experiment", "sensor_channel", "bearing", "axis"],
    how="left",
)

for metric in METRICS:
    thresholds = healthy_thresholds[healthy_thresholds["metric"].eq(metric)][["sensor_channel", "healthy_p95"]]
    scored_features = scored_features.merge(thresholds, on="sensor_channel", how="left")
    scored_features[f"{metric}_above_healthy_p95"] = scored_features[metric].gt(scored_features["healthy_p95"])
    scored_features = scored_features.drop(columns=["healthy_p95"])

metric_evidence_columns = [f"{metric}_above_healthy_p95" for metric in METRICS]
scored_features["degradation_metric_evidence_count"] = scored_features[metric_evidence_columns].sum(axis=1)
scored_features["late_life"] = scored_features["full_life_fraction"].ge(LATE_LIFE_FRACTION)
scored_features["documented_failed_bearing"] = scored_features["bearing"].isin(FAILED_BEARINGS)

scored_features.head()

In [ ]:
def add_persistence_flags(frame: pd.DataFrame, method_column: str, output_column: str) -> pd.DataFrame:
    output = frame.sort_values(["sensor_channel", "timestamp"]).copy()
    persistent_parts = []
    for _, channel_frame in output.groupby("sensor_channel", sort=True):
        flags = channel_frame[method_column].astype(int)
        rolling_alerts = flags.rolling(PERSISTENCE_WINDOW, center=True, min_periods=1).sum()
        channel_frame[output_column] = rolling_alerts.ge(2).to_numpy()
        persistent_parts.append(channel_frame)
    return pd.concat(persistent_parts, ignore_index=True).sort_values(["timestamp", "sensor_channel"])


scored_features = add_persistence_flags(scored_features, "z_is_anomaly", "z_persistent_cluster")
scored_features = add_persistence_flags(scored_features, "if_is_anomaly", "if_persistent_cluster")

scored_features["z_likely_degradation"] = (
    scored_features["z_is_anomaly"]
    & scored_features["late_life"]
    & scored_features["documented_failed_bearing"]
    & scored_features["degradation_metric_evidence_count"].ge(2)
)
scored_features["if_likely_degradation"] = (
    scored_features["if_is_anomaly"]
    & scored_features["late_life"]
    & scored_features["documented_failed_bearing"]
    & scored_features["degradation_metric_evidence_count"].ge(2)
)

scored_features["both_likely_degradation"] = scored_features["z_likely_degradation"] & scored_features["if_likely_degradation"]
scored_features.head()

## 4. Alert Validation Summary

In [ ]:
def summarize_method(frame: pd.DataFrame, method_name: str, alert_column: str, likely_column: str, cluster_column: str) -> dict:
    alerts = frame[frame[alert_column]].copy()
    if alerts.empty:
        return {
            "method": method_name,
            "alerts": 0,
            "likely_degradation_alerts": 0,
            "late_life_alerts": 0,
            "failed_bearing_alerts": 0,
            "persistent_alerts": 0,
            "mean_metric_evidence_count": 0.0,
            "first_alert_timestamp": pd.NaT,
            "first_alert_life_fraction": np.nan,
            "last_alert_timestamp": pd.NaT,
            "last_alert_life_fraction": np.nan,
        }
    return {
        "method": method_name,
        "alerts": int(len(alerts)),
        "likely_degradation_alerts": int(alerts[likely_column].sum()),
        "late_life_alerts": int(alerts["late_life"].sum()),
        "failed_bearing_alerts": int(alerts["documented_failed_bearing"].sum()),
        "persistent_alerts": int(alerts[cluster_column].sum()),
        "mean_metric_evidence_count": float(alerts["degradation_metric_evidence_count"].mean()),
        "first_alert_timestamp": alerts["timestamp"].min(),
        "first_alert_life_fraction": float(alerts.loc[alerts["timestamp"].idxmin(), "full_life_fraction"]),
        "last_alert_timestamp": alerts["timestamp"].max(),
        "last_alert_life_fraction": float(alerts.loc[alerts["timestamp"].idxmax(), "full_life_fraction"]),
    }


validation_summary = pd.DataFrame(
    [
        summarize_method(scored_features, "Dynamic rolling Z-score", "z_is_anomaly", "z_likely_degradation", "z_persistent_cluster"),
        summarize_method(scored_features, "Isolation Forest", "if_is_anomaly", "if_likely_degradation", "if_persistent_cluster"),
    ]
)

validation_summary

In [ ]:
validation_by_bearing = scored_features.groupby("bearing").agg(
    rows=("timestamp", "size"),
    z_alerts=("z_is_anomaly", "sum"),
    z_likely_degradation=("z_likely_degradation", "sum"),
    if_alerts=("if_is_anomaly", "sum"),
    if_likely_degradation=("if_likely_degradation", "sum"),
    both_alerts=("both_anomaly", "sum"),
    both_likely_degradation=("both_likely_degradation", "sum"),
    mean_metric_evidence=("degradation_metric_evidence_count", "mean"),
).reset_index()

validation_by_bearing

## 5. Visual Validation Plots

In [ ]:
def plot_alerts_on_metric(metric: str) -> Path:
    figure, axes = plt.subplots(len(FAILED_BEARINGS), 1, figsize=(15, 8), sharex=True)
    if len(FAILED_BEARINGS) == 1:
        axes = [axes]
    for axis, bearing in zip(axes, sorted(FAILED_BEARINGS)):
        subset = test_features[test_features["bearing"].eq(bearing)]
        sns.lineplot(data=subset, x="timestamp", y=metric, hue="sensor_channel", linewidth=0.9, ax=axis, legend=False)
        z_alerts = scored_features[scored_features["z_likely_degradation"] & scored_features["bearing"].eq(bearing)]
        if_alerts = scored_features[scored_features["if_likely_degradation"] & scored_features["bearing"].eq(bearing)]
        axis.scatter(z_alerts["timestamp"], z_alerts[metric], color="tab:red", s=26, label="Z-score likely degradation")
        axis.scatter(if_alerts["timestamp"], if_alerts[metric], color="black", marker="x", s=36, label="Isolation Forest likely degradation")
        axis.axvspan(test_features["timestamp"].quantile(LATE_LIFE_FRACTION), test_features["timestamp"].max(), color="gold", alpha=0.12)
        axis.set_title(f"Bearing {bearing}: {metric} with validated alerts")
        axis.set_ylabel(metric)
        axis.legend(loc="upper left")
    figure.tight_layout()
    output = PLOTS_DIR / f"ims_validation_{metric}_alerts.png"
    figure.savefig(output, dpi=160, bbox_inches="tight")
    plt.show()
    return output


validation_plot_paths = [plot_alerts_on_metric(metric) for metric in ["rms", "spectral_energy"]]
validation_plot_paths

In [ ]:
heatmap_data = validation_by_bearing.set_index("bearing")[["z_alerts", "z_likely_degradation", "if_alerts", "if_likely_degradation", "both_alerts"]]
figure, axis = plt.subplots(figsize=(9, 4.5))
sns.heatmap(heatmap_data, annot=True, fmt=".0f", cmap="YlOrRd", linewidths=0.5, ax=axis)
axis.set_title("Alert validation counts by bearing")
figure.tight_layout()
heatmap_path = PLOTS_DIR / "ims_validation_alert_counts_by_bearing.png"
figure.savefig(heatmap_path, dpi=160, bbox_inches="tight")
plt.show()

heatmap_path

## 6. Validation Verdict

In [ ]:
def verdict(row: pd.Series) -> str:
    if row["likely_degradation_alerts"] == 0:
        return "weak: alerts do not satisfy the degradation evidence criteria"
    if row["failed_bearing_alerts"] / max(row["alerts"], 1) >= 0.8 and row["late_life_alerts"] / max(row["alerts"], 1) >= 0.8:
        return "strong: alerts mostly occur late and on documented failed bearings"
    return "mixed: some alerts correspond to degradation, but false-positive behavior remains"


validation_summary["validation_verdict"] = validation_summary.apply(verdict, axis=1)
validation_summary

Interpretation guide:

- Alerts marked as likely degradation are late-life alerts on documented failed bearings with at least two degradation metrics above their healthy-channel 95th percentile.
- Dynamic Z-score is expected to catch sharp departures from each channel's recent behavior.
- Isolation Forest is expected to catch multivariate states unlike the early healthy baseline.
- Low overlap means the methods are useful as separate signals, but they should not be integrated as equivalent alert types until more labeled validation is available.

## 7. Save Validation Outputs

In [ ]:
validated_alerts = scored_features[
    scored_features["z_is_anomaly"] | scored_features["if_is_anomaly"]
].copy()

validated_alerts.to_csv(VALIDATED_ALERTS_PATH, index=False)
validation_summary.to_csv(VALIDATION_SUMMARY_PATH, index=False)

saved_outputs = pd.DataFrame(
    [
        {"artifact": "validated alert rows", "path": VALIDATED_ALERTS_PATH.as_posix(), "rows": len(validated_alerts)},
        {"artifact": "validation summary", "path": VALIDATION_SUMMARY_PATH.as_posix(), "rows": len(validation_summary)},
        {"artifact": "validation heatmap", "path": heatmap_path.as_posix(), "rows": None},
        *({"artifact": f"{path.stem} plot", "path": path.as_posix(), "rows": None} for path in validation_plot_paths),
    ]
)

saved_outputs